# NeuroGolf 2026 — Competitive Solver v3

**Speed fixes over v2:**
- CNN trains on `train` pairs only (3-5 examples), NOT 260+ arc-gen pairs -> 50x faster
- Hard 30s CNN timeout; 8 architectures (was 16); `early_fail=200` aborts useless arch in ~0.5s

**Quality fixes:**
- Circular-roll detector + grid-size-aware model
- Combined geom+remap detector (commutativity: one fn handles all 8 geom+remap combos)
- Fixed linear solver (Slice+Gemm+Pad — v2 tried to build 324 MB matrix)
- Validates against ALL examples: train + test + ALL arc-gen

In [1]:
import os, json, math, time, zipfile, warnings
import numpy as np
from pathlib import Path
import onnx, onnxruntime
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto
import torch, torch.nn as nn
import torch.nn.functional as F
import inspect as _inspect
warnings.filterwarnings('ignore')

TASK_DIR   = Path('./')
OUTPUT_DIR = Path('./submission')
SUB_DIR    = OUTPUT_DIR / 'submission'
SUB_DIR.mkdir(parents=True, exist_ok=True)

C, H, W     = 10, 30, 30
OPSET       = oh.make_opsetid('', 17)
IR_VER      = 8
MAX_BYTES   = int(1.44 * 1024 * 1024)
CNN_TIMEOUT = 30.0

DEVICE = (torch.device('cuda') if torch.cuda.is_available() else
          torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cpu'))
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True

Device: mps


In [2]:
def load_task(path):
    with open(path) as f: return json.load(f)

def g2t(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((C, H, W), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            if 0 <= g[r,c] <= 9: t[g[r,c], r, c] = 1.0
    return t

def g2t_batch(pairs, key):
    return np.stack([g2t(p[key]) for p in pairs])

def t2grid(t, oh_, ow_):
    arr = t[0] if t.ndim == 4 else t
    return [
        [int(np.argmax(arr[:, r, c])) if arr[:, r, c].max() > 0.0 else 0
         for c in range(ow_)]
        for r in range(oh_)
    ]

def make_model(nodes, inits=None):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1,C,H,W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1,C,H,W])
    graph = oh.make_graph(nodes, 'g', [X], [Y], initializer=inits or [])
    model = oh.make_model(graph, opset_imports=[OPSET])
    model.ir_version = IR_VER
    return model

def save_onnx(model, path):
    with open(path, 'wb') as f: f.write(model.SerializeToString())
    return Path(path).stat().st_size

def run_onnx(model_or_path, pairs):
    try:
        if isinstance(model_or_path, (str, Path)):
            sess = onnxruntime.InferenceSession(str(model_or_path), providers=['CPUExecutionProvider'])
        else:
            sess = onnxruntime.InferenceSession(model_or_path.SerializeToString(), providers=['CPUExecutionProvider'])
    except Exception: return 0, len(pairs)
    correct = 0
    for p in pairs:
        inp  = g2t(p['input'])[np.newaxis]
        og   = np.array(p['output'])
        oh_, ow_ = og.shape
        try:
            pred = sess.run(None, {'input': inp})[0]
            if t2grid(pred, oh_, ow_) == p['output']: correct += 1
        except Exception: pass
    return correct, len(pairs)

def check_all(model, all_pairs):
    c, t = run_onnx(model, all_pairs)
    return c == t

def score_model(path):
    try:
        import onnx_tool
        m = onnx_tool.loadmodel(str(path), {'verbose': False})
        g = m.graph
        g.graph_reorder_nodes(); g.shape_infer(None); g.profile()
        cost = int(sum(g.macs)) + int(g.memory) + int(g.params)
        return max(1.0, 25.0 - math.log(max(cost, 1)))
    except Exception: return 1.0

def _onnx_export(model, path):
    model.cpu().eval()
    dummy = torch.randn(1, C, H, W)
    kw = dict(opset_version=17, input_names=['input'], output_names=['output'])
    if 'dynamo' in _inspect.signature(torch.onnx.export).parameters:
        kw['dynamo'] = False
    torch.onnx.export(model, dummy, str(path), **kw)

print('Helpers ready.')

Helpers ready.


In [3]:
def _gather_model(indices):
    sh_chw = onh.from_array(np.array([1,C,H*W], np.int64), name='sh_chw')
    sh_out = onh.from_array(np.array([1,C,H,W], np.int64), name='sh_out')
    gi     = onh.from_array(indices.astype(np.int64),       name='gi')
    return make_model([
        oh.make_node('Constant', [], ['sh_chw'], value=sh_chw),
        oh.make_node('Reshape',  ['input','sh_chw'], ['flat']),
        oh.make_node('Constant', [], ['gi'],     value=gi),
        oh.make_node('Gather',   ['flat','gi'],  ['gathered'], axis=2),
        oh.make_node('Constant', [], ['sh_out'], value=sh_out),
        oh.make_node('Reshape',  ['gathered','sh_out'], ['output']),
    ])

def _perm_from_fn(fn):
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            nr, nc = fn(r, c)
            if 0 <= nr < H and 0 <= nc < W:
                idx[r*W+c] = nr*W+nc
    return idx

def model_identity():
    return make_model([oh.make_node('Identity', ['input'], ['output'])])

def model_color_remap(cmap):
    Wt = np.zeros((C,C,1,1), np.float32)
    for s, d in cmap.items(): Wt[d,s,0,0] = 1.0
    for ch in range(C):
        if ch not in cmap: Wt[ch,ch,0,0] = 1.0
    wt = onh.from_array(Wt, name='W')
    return make_model([
        oh.make_node('Constant', [], ['W'], value=wt),
        oh.make_node('Conv', ['input','W'], ['output'], kernel_shape=[1,1], pads=[0,0,0,0]),
    ])

model_rot90   = lambda: _gather_model(_perm_from_fn(lambda r,c:(c,H-1-r)))
model_rot180  = lambda: _gather_model(_perm_from_fn(lambda r,c:(H-1-r,W-1-c)))
model_rot270  = lambda: _gather_model(_perm_from_fn(lambda r,c:(W-1-c,r)))
model_hflip   = lambda: _gather_model(_perm_from_fn(lambda r,c:(r,W-1-c)))
model_vflip   = lambda: _gather_model(_perm_from_fn(lambda r,c:(H-1-r,c)))
model_hvflip  = lambda: _gather_model(_perm_from_fn(lambda r,c:(H-1-r,W-1-c)))
model_transpose    = lambda: _gather_model(_perm_from_fn(lambda r,c:(c,r)))
model_antitranspose= lambda: _gather_model(_perm_from_fn(lambda r,c:(W-1-c,H-1-r)))

def model_scale(factor):
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            idx[r*W+c] = (r//factor)*W + (c//factor)
    return _gather_model(idx)

def model_tile(tr, tc, ih, iw):
    th, tw = ih//tr, iw//tc
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            idx[r*W+c] = (r % th)*W + (c % tw)
    return _gather_model(idx)

def model_crop(r0, c0, oh_, ow_):
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            if r < oh_ and c < ow_:
                sr, sc = r0+r, c0+c
                idx[r*W+c] = sr*W+sc if sr<H and sc<W else 0
    mask = np.zeros((1,C,H,W), np.float32)
    mask[0,:,:oh_,:ow_] = 1.0
    mk   = onh.from_array(mask, name='mask')
    sh_c = onh.from_array(np.array([1,C,H*W],np.int64), name='sh_c')
    sh_o = onh.from_array(np.array([1,C,H,W],np.int64), name='sh_o')
    gi   = onh.from_array(idx, name='gi')
    return make_model([
        oh.make_node('Constant',[],['sh_c'], value=sh_c),
        oh.make_node('Reshape', ['input','sh_c'],['flat']),
        oh.make_node('Constant',[],['gi'],   value=gi),
        oh.make_node('Gather',  ['flat','gi'],['gath'],axis=2),
        oh.make_node('Constant',[],['sh_o'], value=sh_o),
        oh.make_node('Reshape', ['gath','sh_o'],['raw']),
        oh.make_node('Constant',[],['mask'], value=mk),
        oh.make_node('Mul',     ['raw','mask'],['output']),
    ])

def model_const(output_tensor):
    ct = onh.from_array(output_tensor.astype(np.float32), name='ct')
    zr = onh.from_array(np.zeros((1,C,H,W), np.float32),  name='zr')
    return make_model([
        oh.make_node('Constant',[],['ct'],value=ct),
        oh.make_node('Constant',[],['zr'],value=zr),
        oh.make_node('Mul', ['input','zr'],  ['zeroed']),
        oh.make_node('Add', ['zeroed','ct'], ['output']),
    ])

def model_roll(dr, dc, ih, iw):
    idx = np.zeros(H*W, np.int64)
    for r in range(H):
        for c in range(W):
            if r < ih and c < iw:
                idx[r*W+c] = ((r-dr) % ih)*W + ((c-dc) % iw)
    return _gather_model(idx)

def model_remap_geom(cmap, geom_idx):
    Wt   = np.zeros((C,C,1,1), np.float32)
    for s, d in cmap.items(): Wt[d,s,0,0] = 1.0
    for ch in range(C):
        if ch not in cmap: Wt[ch,ch,0,0] = 1.0
    wt   = onh.from_array(Wt, name='W2')
    sh_c = onh.from_array(np.array([1,C,H*W],np.int64), name='sh_c')
    sh_o = onh.from_array(np.array([1,C,H,W],np.int64), name='sh_o')
    gi   = onh.from_array(geom_idx, name='gi2')
    return make_model([
        oh.make_node('Constant',[],['W2'],  value=wt),
        oh.make_node('Conv',['input','W2'],['remapped'],kernel_shape=[1,1],pads=[0,0,0,0]),
        oh.make_node('Constant',[],['sh_c'],value=sh_c),
        oh.make_node('Reshape',['remapped','sh_c'],['flat2']),
        oh.make_node('Constant',[],['gi2'], value=gi),
        oh.make_node('Gather',  ['flat2','gi2'],['gath2'],axis=2),
        oh.make_node('Constant',[],['sh_o'],value=sh_o),
        oh.make_node('Reshape', ['gath2','sh_o'],['output']),
    ])

print('Handcrafted models ready.')

Handcrafted models ready.


In [4]:
def _all_same_size(pairs):
    sizes = set()
    for p in pairs:
        sizes.add((np.array(p['input']).shape, np.array(p['output']).shape))
    return len(sizes) == 1, list(sizes)[0] if len(sizes) == 1 else None

def detect_identity(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok: return False
    return all(np.array_equal(np.array(p['input']), np.array(p['output'])) for p in pairs)

def detect_color_remap(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok or sz[0] != sz[1]: return None
    cmap = {}
    for p in pairs:
        a = np.array(p['input']).flatten()
        b = np.array(p['output']).flatten()
        for s, d in zip(a, b):
            s, d = int(s), int(d)
            if s in cmap and cmap[s] != d: return None
            cmap[s] = d
    return cmap if cmap else None

def detect_rotation(pairs):
    for angle, k in [(90,1),(180,2),(270,3)]:
        if all(np.array_equal(np.rot90(np.array(p['input']),k), np.array(p['output']))
               for p in pairs):
            return angle
    return None

def detect_flip(pairs):
    for ax, name in [(1,'hflip'),(0,'vflip'),(-1,'hvflip')]:
        def flipped(ig, ax=ax):
            return np.flip(np.flip(ig,0),1) if ax==-1 else np.flip(ig,ax)
        if all(np.array_equal(flipped(np.array(p['input'])), np.array(p['output']))
               for p in pairs):
            return name
    return None

def detect_transpose(pairs):
    return all(np.array_equal(np.array(p['input']).T, np.array(p['output'])) for p in pairs)

def detect_antitranspose(pairs):
    return all(np.array_equal(np.fliplr(np.flipud(np.array(p['input']))).T,
                              np.array(p['output'])) for p in pairs)

def detect_scale(pairs):
    for f in [2,3,4,5]:
        if all(np.array_equal(
               np.repeat(np.repeat(np.array(p['input']),f,0),f,1),
               np.array(p['output'])) for p in pairs):
            return f
    return None

def detect_tile(pairs):
    p0 = pairs[0]
    ih,iw    = np.array(p0['input']).shape
    oh_,ow_  = np.array(p0['output']).shape
    if oh_%ih != 0 or ow_%iw != 0: return None
    tr, tc = oh_//ih, ow_//iw
    if all(np.array_equal(np.tile(np.array(p['input']),(tr,tc)),
                          np.array(p['output'])) for p in pairs):
        return tr, tc, ih, iw
    return None

def detect_crop(pairs):
    p0 = pairs[0]
    ig0, og0 = np.array(p0['input']), np.array(p0['output'])
    ih,iw    = ig0.shape
    oh_,ow_  = og0.shape
    if oh_ > ih or ow_ > iw: return None
    for r0 in range(ih-oh_+1):
        for c0 in range(iw-ow_+1):
            if np.array_equal(ig0[r0:r0+oh_, c0:c0+ow_], og0):
                if all(np.array_equal(
                       np.array(p['input'])[r0:r0+oh_, c0:c0+ow_],
                       np.array(p['output'])) for p in pairs[1:]):
                    return r0, c0, oh_, ow_
    return None

def detect_const(pairs):
    if len(set(str(p['output']) for p in pairs)) == 1:
        return g2t(pairs[0]['output'])[np.newaxis]
    return None

def detect_pixel_permutation(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok or sz[0] != sz[1]: return None
    ih, iw = sz[0]
    if ih != H or iw != W: return None
    src = np.array(pairs[0]['input']).flatten()
    dst = np.array(pairs[0]['output']).flatten()
    mapping = {}
    for di, dv in enumerate(dst):
        cands = np.where(src == dv)[0]
        if len(cands) != 1: return None
        mapping[di] = cands[0]
    idx = np.array([mapping[i] for i in range(H*W)], np.int64)
    for p in pairs[1:]:
        s = np.array(p['input']).flatten()
        if not np.array_equal(s[idx], np.array(p['output']).flatten()): return None
    return idx

def detect_roll(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok or sz[0] != sz[1]: return None
    ih, iw = sz[0]
    ig0 = np.array(pairs[0]['input'])
    og0 = np.array(pairs[0]['output'])
    for dr in range(ih):
        for dc in range(iw):
            if dr == 0 and dc == 0: continue
            shifted = np.roll(np.roll(ig0, dr, axis=0), dc, axis=1)
            if np.array_equal(shifted, og0):
                if all(np.array_equal(
                       np.roll(np.roll(np.array(p['input']), dr, 0), dc, 1),
                       np.array(p['output'])) for p in pairs[1:]):
                    return dr, dc, ih, iw
    return None

# -- Combo: geom + color-remap (commutative)
# T(remap(x)) == remap(T(x)) because remap only changes colors, not positions.
# Strategy: apply T to inputs, check if {T(input)->output} is a remap.
_GEOM_FNS = [
    ('rot90',   lambda g: np.rot90(g,1),              lambda: _perm_from_fn(lambda r,c:(c,H-1-r))),
    ('rot180',  lambda g: np.rot90(g,2),              lambda: _perm_from_fn(lambda r,c:(H-1-r,W-1-c))),
    ('rot270',  lambda g: np.rot90(g,3),              lambda: _perm_from_fn(lambda r,c:(W-1-c,r))),
    ('hflip',   lambda g: np.fliplr(g),               lambda: _perm_from_fn(lambda r,c:(r,W-1-c))),
    ('vflip',   lambda g: np.flipud(g),               lambda: _perm_from_fn(lambda r,c:(H-1-r,c))),
    ('hvflip',  lambda g: np.flipud(np.fliplr(g)),    lambda: _perm_from_fn(lambda r,c:(H-1-r,W-1-c))),
    ('transp',  lambda g: g.T,                        lambda: _perm_from_fn(lambda r,c:(c,r))),
    ('atransp', lambda g: np.fliplr(np.flipud(g)).T,  lambda: _perm_from_fn(lambda r,c:(W-1-c,H-1-r))),
]

def detect_geom_remap(pairs):
    ok, sz = _all_same_size(pairs)
    if not ok or sz[0] != sz[1]: return None
    for name, geom_fn, idx_fn in _GEOM_FNS:
        faux, valid = [], True
        for p in pairs:
            ig = np.array(p['input']); og = np.array(p['output'])
            if ig.shape != og.shape: valid = False; break
            try: tr_ig = geom_fn(ig)
            except: valid = False; break
            if tr_ig.shape != og.shape: valid = False; break
            faux.append({'input': tr_ig.tolist(), 'output': og.tolist()})
        if not valid: continue
        cmap = detect_color_remap(faux)
        if cmap and any(cmap.get(i,i) != i for i in range(10)):
            return cmap, name, idx_fn()
    return None

print('Pattern detectors ready.')

Pattern detectors ready.


In [5]:
# Fixed linear solver: Slice+Gemm+Pad (v2 tried to build 324MB full matrix)

def _model_linear_small(W_np, ih, iw, oh_, ow_):
    in_dim  = C * ih * iw
    out_dim = C * oh_ * ow_
    slc_s = onh.from_array(np.array([0,0,0,0],    np.int64), name='slc_s')
    slc_e = onh.from_array(np.array([1,C,ih,iw],  np.int64), name='slc_e')
    sh_fl = onh.from_array(np.array([1,in_dim],   np.int64), name='sh_fl')
    wt    = onh.from_array(W_np.astype(np.float32),           name='W')
    sh_sm = onh.from_array(np.array([1,C,oh_,ow_],np.int64),  name='sh_sm')
    pads  = onh.from_array(np.array([0,0,0,0, 0,0,H-oh_,W-ow_], np.int64), name='pads')
    return make_model([
        oh.make_node('Constant',[],['slc_s'],value=slc_s),
        oh.make_node('Constant',[],['slc_e'],value=slc_e),
        oh.make_node('Slice',['input','slc_s','slc_e'],['sliced']),
        oh.make_node('Constant',[],['sh_fl'],value=sh_fl),
        oh.make_node('Reshape',['sliced','sh_fl'],['flat']),
        oh.make_node('Constant',[],['W'],value=wt),
        oh.make_node('Gemm',['flat','W'],['out_sm'],transB=0,alpha=1.0,beta=0.0),
        oh.make_node('Constant',[],['sh_sm'],value=sh_sm),
        oh.make_node('Reshape',['out_sm','sh_sm'],['small']),
        oh.make_node('Constant',[],['pads'],value=pads),
        oh.make_node('Pad',['small','pads'],['output']),
    ])

def try_linear_solve(all_pairs):
    ok, sz = _all_same_size(all_pairs)
    if not ok: return None
    (ih,iw),(oh_,ow_) = sz
    in_dim  = C * ih * iw
    out_dim = C * oh_ * ow_
    if in_dim * out_dim * 4 > MAX_BYTES: return None
    if len(all_pairs) < 2: return None

    def g2small(grid, h, w):
        t = np.zeros((C, h, w), np.float32)
        for r in range(h):
            for c in range(w):
                t[int(grid[r][c]), r, c] = 1.0
        return t.flatten()

    X = np.stack([g2small(p['input'],  ih, iw)  for p in all_pairs])
    Y = np.stack([g2small(p['output'], oh_, ow_) for p in all_pairs])
    try:
        W, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
        pred = ((X @ W) > 0.0).astype(np.float32)
        if np.max(np.abs(pred - Y)) > 0.05: return None
    except Exception: return None
    return _model_linear_small(W, ih, iw, oh_, ow_)

print('Linear solver ready.')

Linear solver ready.


In [6]:
# Fast trained CNN -- trains on `train` pairs only (3-5 examples)
# v2 trained on all 260+ pairs/epoch -> ~74ms/epoch
# v3 trains on 3-5 pairs/epoch      -> ~0.5ms/epoch on GPU = 50x faster

class ConvNet(nn.Module):
    def __init__(self, kind, hidden=8, k=3, blocks=2):
        super().__init__()
        p = k//2
        if kind == 'conv1':
            self.net = nn.Conv2d(C, C, k, padding=p, bias=False)
        elif kind == 'bneck':
            self.net = nn.Sequential(
                nn.Conv2d(C,hidden,1,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,k,padding=p,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,C,1,bias=False))
        elif kind == 'res':
            layers = [nn.Conv2d(C,hidden,k,padding=p,bias=False), nn.ReLU()]
            for _ in range(blocks-1):
                layers += [nn.Conv2d(hidden,hidden,k,padding=p,bias=False), nn.ReLU()]
            layers.append(nn.Conv2d(hidden,C,k,padding=p,bias=False))
            self.body = nn.Sequential(*layers)
            self.net  = None
        elif kind == 'dilated':
            self.net = nn.Sequential(
                nn.Conv2d(C,hidden,k,padding=p,     dilation=1,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,k,padding=p*2,dilation=2,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,hidden,k,padding=p*4,dilation=4,bias=False), nn.ReLU(),
                nn.Conv2d(hidden,C,1,bias=False))
        self.kind = kind

    def forward(self, x):
        if self.kind == 'res': return x + self.body(x)
        return self.net(x)

ARCH_LIST = [
    ('conv1',   dict(kind='conv1',   k=1)),
    ('conv3',   dict(kind='conv1',   k=3)),
    ('conv5',   dict(kind='conv1',   k=5)),
    ('bnk4k3',  dict(kind='bneck',   hidden=4,  k=3)),
    ('bnk8k3',  dict(kind='bneck',   hidden=8,  k=3)),
    ('bnk8k5',  dict(kind='bneck',   hidden=8,  k=5)),
    ('res8b2',  dict(kind='res',     hidden=8,  k=3, blocks=2)),
    ('dil8',    dict(kind='dilated', hidden=8,  k=3)),
]

def train_conv(train_pairs, arch_kwargs,
               max_epochs=2000, lr=0.01, patience=200, early_fail=200,
               timeout=CNN_TIMEOUT):
    t0 = time.time()
    for p in train_pairs:
        if np.array(p['input']).shape != np.array(p['output']).shape:
            return None, False
    X = torch.tensor(g2t_batch(train_pairs,'input')).to(DEVICE)
    Y = torch.tensor(g2t_batch(train_pairs,'output')).to(DEVICE)
    n = len(train_pairs)
    model = ConvNet(**arch_kwargs).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sch   = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=500, T_mult=2)
    best_correct, best_state, no_imp = 0, None, 0

    def accuracy():
        model.eval()
        with torch.no_grad():
            pred = (model(X) > 0).float()
            return int(torch.all(pred == Y, dim=(1,2,3)).sum().item())

    for epoch in range(max_epochs):
        if time.time() - t0 > timeout: break
        model.train()
        opt.zero_grad()
        out  = model(X)
        loss = F.mse_loss(out, Y) + F.binary_cross_entropy_with_logits(out * 5, Y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
        if (epoch+1) % 25 == 0:
            c = accuracy()
            if (epoch+1) >= early_fail and best_correct == 0: return None, False
            if c > best_correct:
                best_correct = c
                best_state   = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 25
            if c == n: break
            if no_imp >= patience: break

    if best_state is None or best_correct < n: return None, False
    model.load_state_dict(best_state)
    model.cpu()
    return model, True

print('Trained conv solver ready.')

Trained conv solver ready.


In [7]:
def solve_task(task_data, task_num):
    train  = task_data.get('train',   [])
    test   = task_data.get('test',    [])
    arcgen = task_data.get('arc-gen', [])
    all_p  = train + test + arcgen
    tr_p   = train + test

    def accept(model, tier):
        if check_all(model, all_p): return model, tier
        return None, None

    if detect_identity(tr_p):
        r = accept(model_identity(), 'identity')
        if r[0]: return r

    cmap = detect_color_remap(tr_p)
    if cmap:
        r = accept(model_color_remap(cmap), 'color_remap')
        if r[0]: return r

    rot = detect_rotation(tr_p)
    if rot:
        mfn = {90:model_rot90, 180:model_rot180, 270:model_rot270}[rot]
        r = accept(mfn(), f'rot{rot}')
        if r[0]: return r

    fl = detect_flip(tr_p)
    if fl:
        mfn = {'hflip':model_hflip, 'vflip':model_vflip, 'hvflip':model_hvflip}[fl]
        r = accept(mfn(), fl)
        if r[0]: return r

    if detect_transpose(tr_p):
        r = accept(model_transpose(), 'transpose')
        if r[0]: return r
    if detect_antitranspose(tr_p):
        r = accept(model_antitranspose(), 'antitranspose')
        if r[0]: return r

    sc = detect_scale(tr_p)
    if sc:
        r = accept(model_scale(sc), f'scale{sc}x')
        if r[0]: return r

    tile = detect_tile(tr_p)
    if tile:
        r = accept(model_tile(*tile), f'tile{tile[:2]}')
        if r[0]: return r

    crop = detect_crop(tr_p)
    if crop:
        r = accept(model_crop(*crop), f'crop{crop[:2]}')
        if r[0]: return r

    roll = detect_roll(tr_p)
    if roll:
        r = accept(model_roll(*roll), f'roll{roll[:2]}')
        if r[0]: return r

    perm = detect_pixel_permutation(tr_p)
    if perm is not None:
        r = accept(_gather_model(perm), 'pixel_perm')
        if r[0]: return r

    ct = detect_const(tr_p)
    if ct is not None:
        r = accept(model_const(ct), 'const_output')
        if r[0]: return r

    gr = detect_geom_remap(tr_p)
    if gr:
        cmap2, name2, idx2 = gr
        r = accept(model_remap_geom(cmap2, idx2), f'remap+{name2}')
        if r[0]: return r

    lin = try_linear_solve(all_p)
    if lin:
        r = accept(lin, 'linear')
        if r[0]: return r

    # CNN tier: train on `train` only, validate on all_p
    sizes = set()
    for p in tr_p:
        ih,iw   = np.array(p['input']).shape
        oh_,ow_ = np.array(p['output']).shape
        sizes.add((ih,iw,oh_,ow_))
    if len(sizes) == 1:
        ih,iw,oh_,ow_ = next(iter(sizes))
        if ih == oh_ and iw == ow_:
            t_cnn = time.time()
            for arch_name, arch_kw in ARCH_LIST:
                remaining = CNN_TIMEOUT - (time.time()-t_cnn)
                if remaining < 1.0: break
                model_t, success = train_conv(train, arch_kw, timeout=remaining)
                if not success: continue
                tmp = SUB_DIR / f'_tmp_{task_num}.onnx'
                try:
                    _onnx_export(model_t, tmp)
                    if tmp.stat().st_size > MAX_BYTES: continue
                    passed, total = run_onnx(tmp, all_p)
                    if passed == total:
                        m = onnx.load(str(tmp))
                        return m, f'cnn_{arch_name}'
                except Exception: pass
                finally:
                    if tmp.exists(): tmp.unlink()

    return None, None

print('Master solver ready.')

Master solver ready.


In [8]:
task_files = sorted(TASK_DIR.glob('task*.json'))
print(f'Found {len(task_files)} tasks')

results, solved, tot_score = [], 0, 0.0
t_start = time.time()

for tf in task_files:
    task_num = int(tf.stem.replace('task',''))
    t0 = time.time()
    try:
        task_data   = load_task(tf)
        model, tier = solve_task(task_data, task_num)
    except Exception:
        model, tier = None, None
    dt = time.time() - t0

    if model is not None:
        out_path = SUB_DIR / f'task{task_num:03d}.onnx'
        save_onnx(model, out_path)
        sc = score_model(out_path)
        solved += 1; tot_score += sc
        tag = f'OK {tier[:14]:14s}  {sc:.1f}pts  {dt:.1f}s'
    else:
        tag = f'-- unsolved              {dt:.1f}s'

    results.append(dict(task=task_num, tier=tier, solved=(model is not None), score=round(tot_score,2)))

    if model is not None or task_num % 10 == 0:
        elapsed = time.time()-t_start
        eta = elapsed/task_num*(400-task_num) if task_num > 0 else 0
        print(f'[{task_num:3d}] {tag}  | {solved}/{task_num} solved  total={tot_score:.1f}  ETA={eta/60:.1f}m')

elapsed = time.time()-t_start
print(f'\n{"="*65}')
print(f'Solved: {solved}/400  Score: {tot_score:.1f}  Time: {elapsed:.0f}s ({elapsed/60:.1f}m)')
print(f'{"="*65}')

Found 400 tasks
[ 10] -- unsolved              2.9s  | 0/10 solved  total=0.0  ETA=7.9m
[ 16] OK color_remap     1.0pts  0.0s  | 1/16 solved  total=1.0  ETA=8.1m
[ 20] -- unsolved              2.6s  | 1/20 solved  total=1.0  ETA=8.1m
[ 30] -- unsolved              0.0s  | 1/30 solved  total=1.0  ETA=6.3m
[ 40] -- unsolved              2.7s  | 1/40 solved  total=1.0  ETA=6.5m
[ 50] -- unsolved              0.0s  | 1/50 solved  total=1.0  ETA=7.0m
[ 53] OK roll(1, 0)      1.0pts  0.0s  | 2/53 solved  total=2.0  ETA=6.8m
[ 60] OK linear          1.0pts  0.4s  | 3/60 solved  total=3.0  ETA=6.5m
[ 70] -- unsolved              3.3s  | 3/70 solved  total=3.0  ETA=6.5m
[ 73] OK cnn_conv5       1.0pts  0.8s  | 4/73 solved  total=4.0  ETA=6.5m
[ 80] -- unsolved              0.1s  | 4/80 solved  total=4.0  ETA=6.3m
[ 87] OK linear          1.0pts  0.1s  | 5/87 solved  total=5.0  ETA=5.9m
[ 90] -- unsolved              0.0s  | 5/90 solved  total=5.0  ETA=5.8m
[ 95] OK cnn_conv3       1.0pts  0.5s 

KeyboardInterrupt: 

In [ ]:
from collections import Counter
tier_counts = Counter(r['tier'] for r in results if r['tier'])
print('Solutions by tier:')
for t, c in tier_counts.most_common(25):
    print(f'  {t:25s}: {c}')

onnx_files = sorted(SUB_DIR.glob('task*.onnx'))
zip_path   = OUTPUT_DIR / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in onnx_files: zf.write(f, f.name)

print(f'\nsubmission.zip: {len(onnx_files)} files, {zip_path.stat().st_size/1024:.0f} KB')
with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Done -- submit /kaggle/working/submission.zip')